# People Analytics: Decoding Workforce Attrition
## Step 1 — Data Cleaning & Preparation

---

I built this project because during my time in HR, I noticed that attrition discussions always happened **AFTER** people resigned — never before. Every exit interview felt like a post-mortem. Leadership would ask "why are people leaving?" and we'd scramble to pull together anecdotal evidence from managers. But the data was right there — in our HRIS, in the engagement surveys, in the overtime logs. Nobody was looking at it proactively.

This project is my attempt to do what I wished we'd done: treat employee attrition like a **business problem with measurable drivers**, not just an HR headache. I'm using the IBM HR Analytics dataset (1,470 employee records, 35 features) as a proxy for the kind of workforce data most mid-size companies already have.

In this notebook, I clean and prepare the data for analysis. Every step is documented with the *why*, not just the *what* — because data cleaning decisions should be defensible, especially when the findings will be presented to leadership.

## 1. Loading the Dataset

The IBM HR Analytics Employee Attrition & Performance dataset is a synthetic dataset published by IBM data scientists. It mirrors the kind of data you'd find in a real HRIS export — demographics, compensation, role details, and satisfaction survey scores. 1,470 records is a realistic size for a single business unit or mid-size company.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Display settings — I want to see all columns when I inspect the data,
# not get a truncated view that hides potential issues
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

print("Libraries loaded successfully.")

In [ ]:
# Load the raw dataset
# Using a public GitHub mirror of the IBM HR Attrition dataset.
# If this URL doesn't work, download from Kaggle and place in data/ folder.

dataset_url = "https://raw.githubusercontent.com/nelson-wu/employee-attrition-ml/master/WA_Fn-UseC_-HR-Employee-Attrition.csv"

try:
    hr_raw = pd.read_csv(dataset_url)
    print(f"Dataset loaded from URL: {hr_raw.shape[0]} rows, {hr_raw.shape[1]} columns")
    # Save a local copy so we're not dependent on the URL later
    hr_raw.to_csv('data/WA_Fn-UseC_-HR-Employee-Attrition.csv', index=False)
    print("Local copy saved to data/ folder.")
except Exception as e:
    print(f"URL download failed ({e}). Loading from local file...")
    hr_raw = pd.read_csv('data/WA_Fn-UseC_-HR-Employee-Attrition.csv')
    print(f"Dataset loaded locally: {hr_raw.shape[0]} rows, {hr_raw.shape[1]} columns")

In [ ]:
# First look — I always start with .head() to see actual data values,
# not just column names. You catch encoding issues and weird values here.

hr_raw.head()

In [ ]:
# .info() tells me three things at once:
# 1. Which columns have missing values (non-null counts < 1470)
# 2. Data types — are numbers stored as strings? Are categories coded as integers?
# 3. Memory usage — not critical for 1,470 rows, but good habit

hr_raw.info()

In [ ]:
# .describe() for numeric columns — looking for:
# - Suspicious min/max values (negative ages? salaries of 0?)
# - Columns with zero variance (same value for everyone — useless for analysis)
# - Income distributions that might need binning later

hr_raw.describe().round(1)

## 2. Data Quality Checks

In my HR experience, data quality issues usually fall into three buckets:
1. **Missing data** — common in HRIS exports, especially for newer hires or recently added fields
2. **Duplicates** — happens when data is merged from multiple systems (e.g., payroll + engagement survey)
3. **Constant columns** — fields that were added to the system but never actually used

Let's check all three.

In [ ]:
# Check for missing values across all columns
# In real HRIS data, you'd typically see nulls in fields like
# YearsSinceLastPromotion (new hires) or ManagerRating (pending reviews)

missing_values = hr_raw.isnull().sum()
missing_pct = (hr_raw.isnull().sum() / len(hr_raw) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

print("=== Missing Value Report ===")
print(f"Total columns: {len(missing_report)}")
print(f"Columns with missing data: {(missing_values > 0).sum()}")
print()
missing_report[missing_report['Missing Count'] > 0] if (missing_values > 0).any() else print("No missing values found — this is a clean synthetic dataset.")

In [ ]:
# Check for duplicate rows
# EmployeeNumber should be unique — if it's not, we have a data merge problem

total_duplicates = hr_raw.duplicated().sum()
employee_id_duplicates = hr_raw['EmployeeNumber'].duplicated().sum()

print(f"Duplicate rows (all columns): {total_duplicates}")
print(f"Duplicate EmployeeNumbers: {employee_id_duplicates}")

# Good — no duplicates. In a real-world scenario, I'd also check for
# near-duplicates (same name + department but different employee number),
# which can happen during system migrations.

In [ ]:
# Find columns with only one unique value (zero variance)
# These are useless for analysis — they don't differentiate anyone

constant_cols = [col for col in hr_raw.columns if hr_raw[col].nunique() == 1]

print("Columns with only one unique value (will be dropped):")
for col in constant_cols:
    print(f"  - {col}: always '{hr_raw[col].iloc[0]}'")

# EmployeeCount = 1 for everyone (it's a per-row record, obviously count is 1)
# Over18 = 'Y' for everyone (legal/compliance field, no minors in dataset)
# StandardHours = 80 for everyone (company-wide policy, not individual)
# These are artifacts of the HRIS system, not meaningful features

## 3. Cleaning & Transformation

Now comes the opinionated part. Every cleaning decision here reflects a judgment call — I'll explain the reasoning so anyone reviewing this can agree or push back.

In [ ]:
# Drop constant-value columns — they add nothing to analysis
# Also dropping EmployeeNumber: it's an ID, not a feature.
# In a real project, I'd keep it for joining tables, but here
# we're working with a single flat file.

columns_to_drop = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']

hr_clean = hr_raw.drop(columns=columns_to_drop)

print(f"Dropped {len(columns_to_drop)} columns: {columns_to_drop}")
print(f"Remaining columns: {hr_clean.shape[1]}")

In [ ]:
# Look at the categorical columns and their unique values
# This helps me understand what I'm working with before encoding anything

categorical_cols = hr_clean.select_dtypes(include='object').columns.tolist()

print("=== Categorical Columns ===")
for col in categorical_cols:
    unique_vals = hr_clean[col].unique()
    print(f"\n{col} ({len(unique_vals)} unique):")
    print(f"  Values: {list(unique_vals)}")

In [ ]:
# Some numeric columns are actually ordinal survey scores (1-4 or 1-5 scale)
# They're stored as integers but represent categories like "Low / Medium / High / Very High"
# I won't convert these to categorical type because:
# 1. For SQL queries and correlation analysis, keeping them numeric is practical
# 2. The ordinal relationship (1 < 2 < 3 < 4) is meaningful
# But I WILL document what the scores mean for anyone reading this

ordinal_mapping = {
    'Education': {1: 'Below College', 2: 'College', 3: 'Bachelor', 4: 'Master', 5: 'Doctor'},
    'EnvironmentSatisfaction': {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'JobInvolvement': {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'JobSatisfaction': {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'PerformanceRating': {1: 'Low', 2: 'Good', 3: 'Excellent', 4: 'Outstanding'},
    'RelationshipSatisfaction': {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'WorkLifeBalance': {1: 'Bad', 2: 'Good', 3: 'Better', 4: 'Best'}
}

print("Ordinal columns kept as integers for analysis.")
print("Reference mapping documented above for interpretation.")
print(f"\nExample — JobSatisfaction value distribution:")
print(hr_clean['JobSatisfaction'].value_counts().sort_index())

## 4. Creating Derived Columns

Two new columns that will be essential for the rest of the analysis:

1. **AttritionFlag** — numeric version of the Yes/No target. Makes aggregations and calculations cleaner.
2. **TenureBand** — groups employees by how long they've been with the company. This is how HR actually thinks about turnover: "Are we losing people in their first two years?" is a fundamentally different problem than "Are 10-year veterans burning out?"

In [ ]:
# AttritionFlag: Yes → 1, No → 0
# This makes calculating attrition rates trivial: just take the mean
# mean(AttritionFlag) = attrition rate as a decimal

hr_clean['AttritionFlag'] = hr_clean['Attrition'].map({'Yes': 1, 'No': 0})

# Quick sanity check — does the flag match the original?
attrition_check = hr_clean.groupby('Attrition')['AttritionFlag'].mean()
print("Sanity check (should be Yes=1.0, No=0.0):")
print(attrition_check)
print(f"\nOverall attrition rate: {hr_clean['AttritionFlag'].mean():.1%}")

In [ ]:
# TenureBand: grouping YearsAtCompany into meaningful HR categories
#
# Why these specific bands?
# - 0-2 years ("Early"): The onboarding/ramp-up phase. Attrition here
#   usually signals poor hiring, bad onboarding, or expectation mismatch.
# - 3-5 years ("Mid"): Employees have hit their stride. Attrition here
#   often means lack of growth or internal mobility.
# - 6+ years ("Senior"): Institutional knowledge holders. Losing them
#   is the most expensive — they take relationships and context with them.

def assign_tenure_band(years):
    if years <= 2:
        return 'Early (0-2 yrs)'
    elif years <= 5:
        return 'Mid (3-5 yrs)'
    else:
        return 'Senior (6+ yrs)'

hr_clean['TenureBand'] = hr_clean['YearsAtCompany'].apply(assign_tenure_band)

# Check distribution — does this look reasonable?
tenure_summary = hr_clean.groupby('TenureBand').agg(
    employee_count=('AttritionFlag', 'count'),
    attrition_rate=('AttritionFlag', 'mean')
).round(3)

tenure_summary['attrition_rate'] = tenure_summary['attrition_rate'].apply(lambda x: f"{x:.1%}")

print("Tenure Band Summary:")
print(tenure_summary)
print("\nThis confirms a pattern I've seen in practice: early-tenure employees")
print("leave at higher rates. The question is whether it's a hiring problem or")
print("an onboarding problem — the data alone won't tell us that.")

## 5. Final Review & Export

Before saving, let's do a final check on the cleaned dataset.

In [ ]:
# Final dataset overview
print("=== Cleaned Dataset Summary ===")
print(f"Rows: {hr_clean.shape[0]}")
print(f"Columns: {hr_clean.shape[1]}")
print(f"Columns added: AttritionFlag, TenureBand")
print(f"Columns dropped: EmployeeCount, Over18, StandardHours, EmployeeNumber")
print(f"\nColumn list:")
for i, col in enumerate(hr_clean.columns, 1):
    dtype = hr_clean[col].dtype
    nunique = hr_clean[col].nunique()
    print(f"  {i:2d}. {col:<30s} {str(dtype):<10s} ({nunique} unique)")

In [ ]:
# Quick attrition breakdown to carry forward into analysis
# This gives us the baseline numbers we'll reference throughout the project

total_employees = len(hr_clean)
left_count = hr_clean['AttritionFlag'].sum()
stayed_count = total_employees - left_count
attrition_rate = left_count / total_employees

print("=== Attrition Baseline ===")
print(f"Total employees: {total_employees:,}")
print(f"Left the company: {left_count:,} ({attrition_rate:.1%})")
print(f"Still employed: {stayed_count:,} ({1 - attrition_rate:.1%})")
print(f"\nFor context: industry average voluntary turnover is typically")
print(f"12-15% annually. Our dataset shows {attrition_rate:.1%}, which is")
print(f"{'above' if attrition_rate > 0.15 else 'within'} that benchmark.")

In [ ]:
# Save the cleaned dataset
# Using index=False because the row index isn't meaningful data

output_path = 'output/hr_cleaned.csv'
hr_clean.to_csv(output_path, index=False)

# Verify the save by reading it back
verify = pd.read_csv(output_path)
print(f"Cleaned data saved to: {output_path}")
print(f"Verification — loaded back: {verify.shape[0]} rows, {verify.shape[1]} columns")
print(f"\nThis file is used by:")
print(f"  - hr_analysis.sql (loaded into SQLite for querying)")
print(f"  - hr_eda.ipynb (EDA visualizations)")
print(f"  - Tableau dashboard (data source)")

---

### What's Next

The cleaned dataset (`hr_cleaned.csv`) is ready for:
1. **SQL Analysis** (`hr_analysis.sql`) — Business questions answered with structured queries
2. **Exploratory Data Analysis** (`hr_eda.ipynb`) — Visual patterns and statistical relationships  
3. **Tableau Dashboard** — Executive-ready interactive views

One thing I'm already noticing from the summary stats: the median monthly income is around $4,919, but the mean is higher — suggesting a right-skewed distribution. A few high earners are pulling the average up. I'll want to look at whether attrition patterns differ by income band in the EDA.